# Amazon Data Contract Validation

This notebook explains and validates the canonical contracts for Amazon reviews, products, and their enriched join. A data contract defines which fields downstream code may rely on, their meaning, their allowed values, and their relationship to source data.

## Objectives

- verify versioned review, product, and enriched-review contracts;
- validate the complete canonical review artifact at its physical boundary;
- demonstrate structured Pydantic errors on invalid logical records;
- explain the different roles of `asin` and `parent_asin`;
- join one real review to real Amazon product metadata without creating a sample artifact.

# Проверка контрактов Amazon-данных

Этот ноутбук объясняет и проверяет канонические контракты для Amazon-отзывов, товаров и результата их соединения. Контракт данных — это явное соглашение о том, какие поля гарантированы последующим этапам, что они означают, какие значения допустимы и как запись связана с исходником.

## Цели

- проверить версионные контракты отзыва, товара и обогащенного отзыва;
- проверить полный канонический датасет отзывов на физической границе;
- показать структурированные ошибки Pydantic на некорректных записях;
- объяснить разницу между `asin` и `parent_asin`;
- соединить реальный отзыв с реальными метаданными товара, не создавая лишний sample-файл.

## Inputs and outputs

Inputs are the registered dataset manifest, the canonical review Parquet, and the immutable compressed Amazon product-metadata source. Reusable contracts live in `src/schemas/`; source adaptation lives in `src/ingestion/`.

The notebook produces only displayed validation results. It intentionally does not persist another sample dataset; the full product catalog will be a separate versioned artifact in the catalog stage.

## Входы и результаты

На вход подаются зарегистрированный манифест датасета, канонический Parquet с отзывами и неизменяемый сжатый Amazon-файл с метаданными товаров. Переиспользуемые контракты находятся в `src/schemas/`, а преобразование исходных записей — в `src/ingestion/`.

Ноутбук выводит результаты проверки только на экран. Он специально не сохраняет еще один выборочный датасет: полный каталог товаров создается как отдельный версионный артефакт.

In [ ]:
# Standard library / Стандартная библиотека
import gzip
import json
import sys
from pathlib import Path

# Third-party packages / Сторонние библиотеки
import duckdb
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from pydantic import ValidationError

# Locate the repository root before importing local project modules.
# Находим корень репозитория до импорта локальных модулей проекта.
PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "PLAN.md").is_file() and (candidate / "src").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Local project modules / Локальные модули проекта
from src.common.project import find_project_root
from src.ingestion.amazon_records import transform_amazon_product_record
from src.ingestion.dataset_manifest import load_dataset_manifest
from src.preprocessing.reviews import (
    CANONICAL_REVIEW_SCHEMA,
    validate_canonical_review_parquet,
)
from src.schemas.amazon import (
    AmazonEnrichedReviewRecord,
    AmazonProductRecord,
    AmazonReviewRecord,
)

In [ ]:
# Resolve all paths from one registered dataset version.
# Получаем все пути из одной зарегистрированной версии датасета.
PROJECT_ROOT = find_project_root(PROJECT_ROOT)
MANIFEST_PATH = (
    PROJECT_ROOT
    / "config/datasets/amazon_reviews_2023_beauty_2021_2023_v1.json"
)
manifest = load_dataset_manifest(MANIFEST_PATH)
CANONICAL_REVIEWS_PATH = (
    PROJECT_ROOT / manifest.file_by_role("canonical_reviews").path
)
RAW_METADATA_PATH = (
    PROJECT_ROOT / manifest.file_by_role("raw_product_metadata").path
)
SOURCE_PRODUCT_SAMPLE_SIZE = 5
CANONICAL_REVIEW_SAMPLE_SIZE = 5

print(f"Python executable: {sys.executable}")
print(f"DuckDB version: {duckdb.__version__}")
print(f"Dataset version: {manifest.dataset_version}")
print(f"Dataset category: {manifest.dataset_category}")

## 1. Contract identity and versioning

A dataset version identifies a fixed population of data. A schema version identifies the structure and meaning expected for one record type. They change for different reasons: adding new source data changes the dataset version, while changing a required field or its meaning changes the schema version.

## Идентичность и версии контрактов

Версия датасета определяет фиксированную совокупность данных. Версия схемы определяет структуру и смысл одного типа записи. Они меняются по разным причинам: добавление новых исходных данных меняет версию датасета, а изменение обязательного поля или его смысла — версию схемы.

In [ ]:
contract_registry = pd.DataFrame(
    [
        {
            "record_type": "canonical review",
            "schema_version": manifest.schema_versions.canonical_reviews,
            "python_contract": AmazonReviewRecord.__name__,
            "artifact_status": "full artifact exists",
        },
        {
            "record_type": "Amazon product",
            "schema_version": manifest.schema_versions.amazon_products,
            "python_contract": AmazonProductRecord.__name__,
            "artifact_status": "full product catalog exists",
        },
        {
            "record_type": "enriched review",
            "schema_version": manifest.schema_versions.enriched_reviews,
            "python_contract": AmazonEnrichedReviewRecord.__name__,
            "artifact_status": "validated deterministic join; not materialized",
        },
    ]
)
display(contract_registry)

## 2. Physical review artifact

The physical boundary is the Parquet file read by downstream code. We compare every column name, Arrow type, and nullability flag with the canonical schema, then scan all retained rows for nulls in mandatory fields. Both the declared physical schema and the observed values must satisfy the contract.

## Физический артефакт отзывов

Физическая граница — это Parquet-файл, который читает последующий код. Мы сравниваем все названия колонок, Arrow-типы и флаги nullability с канонической схемой, затем проверяем все сохраненные строки на null в обязательных полях. Контракт должны одновременно выполнять и объявленная физическая схема, и фактические значения.

In [ ]:
artifact_validation = validate_canonical_review_parquet(
    CANONICAL_REVIEWS_PATH
)
physical_schema = pq.ParquetFile(CANONICAL_REVIEWS_PATH).schema_arrow
schema_table = pd.DataFrame(
    {
        "column": CANONICAL_REVIEW_SCHEMA.names,
        "expected_type": [
            str(field.type) for field in CANONICAL_REVIEW_SCHEMA
        ],
        "physical_type": [str(field.type) for field in physical_schema],
        "expected_nullable": [
            field.nullable for field in CANONICAL_REVIEW_SCHEMA
        ],
        "physical_nullable": [
            field.nullable for field in physical_schema
        ],
        "contract_required": [
            not field.nullable for field in CANONICAL_REVIEW_SCHEMA
        ],
        "actual_null_count": [
            artifact_validation.required_null_counts.get(field.name)
            for field in CANONICAL_REVIEW_SCHEMA
        ],
    }
)

assert artifact_validation.is_valid
assert artifact_validation.row_count == (
    manifest.file_by_role("canonical_reviews").record_count
)
display(
    pd.Series(
        {
            "row_count": artifact_validation.row_count,
            "column_names_match": artifact_validation.column_names_match,
            "column_types_match": artifact_validation.column_types_match,
            "column_nullability_match": (
                artifact_validation.column_nullability_match
            ),
            "contract_passed": artifact_validation.is_valid,
        },
        name="value",
    ).to_frame()
)
display(schema_table)

## 3. Logical review validation

Arrow validates storage types efficiently for millions of rows. Pydantic adds business rules such as rating range, non-empty identifiers, non-negative helpful votes, UTC-aware timestamps, and rejection of unknown fields. We validate a readable batch with Pydantic while the previous step checks the complete artifact physically.

## Логическая проверка отзывов

Arrow эффективно проверяет типы хранения для миллионов строк. Pydantic добавляет бизнес-правила: допустимый рейтинг, непустые идентификаторы, неотрицательные helpful votes, время с часовым поясом UTC и запрет неизвестных полей. Небольшую понятную выборку мы проверяем через Pydantic, а предыдущий этап проверяет физический контракт всего артефакта.

In [ ]:
review_batch = (
    pq.ParquetFile(CANONICAL_REVIEWS_PATH)
    .read_row_group(0)
    .slice(0, CANONICAL_REVIEW_SAMPLE_SIZE)
    .to_pylist()
)
validated_reviews = [
    AmazonReviewRecord.model_validate(record) for record in review_batch
]
assert len(validated_reviews) == CANONICAL_REVIEW_SAMPLE_SIZE
display(
    pd.DataFrame(
        [review.model_dump() for review in validated_reviews]
    )[
        [
            "review_id",
            "asin",
            "parent_asin",
            "rating",
            "review_timestamp",
            "review_text",
        ]
    ]
)

In [ ]:
# Deliberately damage one valid record in several independent ways.
# Намеренно портим одну корректную запись несколькими независимыми способами.
valid_payload = validated_reviews[0].model_dump()
invalid_cases = [
    ("rating outside 1–5", valid_payload | {"rating": 7}),
    ("empty parent_asin", valid_payload | {"parent_asin": ""}),
    ("naive timestamp", valid_payload | {"review_timestamp": "2023-01-01T00:00:00"}),
    ("unknown field", valid_payload | {"invented_metric": 99}),
]
validation_errors = []
for case_name, payload in invalid_cases:
    try:
        AmazonReviewRecord.model_validate(payload)
    except ValidationError as error:
        for detail in error.errors(include_url=False):
            validation_errors.append(
                {
                    "case": case_name,
                    "field": ".".join(str(item) for item in detail["loc"]),
                    "error_type": detail["type"],
                    "message": detail["msg"],
                }
            )

assert len(validation_errors) == len(invalid_cases)
display(pd.DataFrame(validation_errors))

## 4. Amazon product contract

The item-metadata source is separate from reviews. It contains one product-family record keyed by `parent_asin`, structured category paths, listing text, rating metadata, price at collection time, images, videos, and details. Missing optional values are retained as null or empty collections and recorded as quality flags instead of inventing replacements.

## Контракт товара Amazon

Метаданные товаров хранятся отдельно от отзывов. В них одна запись семейства товара определяется через `parent_asin` и содержит путь категорий, текст листинга, агрегированный рейтинг, цену на момент сбора, изображения, видео и details. Отсутствующие необязательные значения сохраняются как null или пустые коллекции и отмечаются quality flags — мы не придумываем подстановки.

In [ ]:
source_products = []
with gzip.open(RAW_METADATA_PATH, "rt", encoding="utf-8") as stream:
    for source_record_index in range(SOURCE_PRODUCT_SAMPLE_SIZE):
        source_products.append(json.loads(next(stream)))

validated_products = [
    transform_amazon_product_record(
        source,
        dataset_version=manifest.dataset_version,
        dataset_category=manifest.dataset_category,
        source_record_index=index,
    )
    for index, source in enumerate(source_products)
]
product_table = pd.DataFrame(
    [
        {
            "source_record_index": product.source_record_index,
            "parent_asin": product.parent_asin,
            "product_title": product.product_title,
            "category_leaf": (
                product.category_path[-1] if product.category_path else None
            ),
            "store": product.store,
            "price_at_collection": product.price_at_collection,
            "metadata_quality_flags": product.metadata_quality_flags,
        }
        for product in validated_products
    ]
)
display(product_table)

## 5. `asin` versus `parent_asin`

`asin` identifies the reviewed variation—for example a specific size, color, or package. `parent_asin` identifies the product family and is the key available in the item-metadata file. The default seller analytics unit is `parent_asin`, but retaining `asin` allows later variation-level analysis and prevents information loss.

The enriched contract rejects a join when dataset identity, category, or `parent_asin` differs. This prevents attaching a review to the wrong product.

## Разница идентификаторов

`asin` определяет конкретный вариант, на который написан отзыв: например размер, цвет или упаковку. `parent_asin` определяет семейство товара и является ключом в файле метаданных. По умолчанию аналитика продавца строится на уровне `parent_asin`, но сохранение `asin` позволяет позже анализировать варианты и не терять информацию.

Контракт обогащенного отзыва отклоняет соединение, если различаются версия датасета, категория или `parent_asin`. Так отзыв нельзя незаметно прикрепить к неправильному товару.

In [ ]:
join_product = validated_products[0]
connection = duckdb.connect()
try:
    review_frame = connection.execute(
        """
        SELECT *
        FROM read_parquet(?)
        WHERE parent_asin = ?
        ORDER BY source_record_index
        LIMIT 1
        """,
        [str(CANONICAL_REVIEWS_PATH), join_product.parent_asin],
    ).fetchdf()
finally:
    connection.close()

if review_frame.empty:
    raise ValueError("The selected real product has no canonical review")
join_review = AmazonReviewRecord.model_validate(review_frame.iloc[0].to_dict())
enriched_record = AmazonEnrichedReviewRecord(
    review=join_review, product=join_product
)

assert enriched_record.review.parent_asin == enriched_record.product.parent_asin
display(
    pd.Series(
        {
            "review_id": enriched_record.review.review_id,
            "variation_asin": enriched_record.review.asin,
            "product_parent_asin": enriched_record.product.parent_asin,
            "product_title": enriched_record.product.product_title,
            "category_path": " > ".join(enriched_record.product.category_path),
            "rating": enriched_record.review.rating,
            "review_text": enriched_record.review.review_text,
        },
        name="value",
    ).to_frame()
)

## 6. Conclusion and limitations

The full canonical review artifact satisfies the registered `amazon_review_v1` physical contract, and representative logical records satisfy the strict Pydantic model. Real Amazon product metadata can be adapted to `amazon_product_v1`, and the enriched contract protects the `parent_asin` join. No sample artifact was written.

The complete versioned product catalog and category registry now exist and are validated in notebooks 04 and 05. A separate full enriched-review artifact is not persisted yet because the join is deterministic; later offline ML requirements will determine whether materializing that additional copy is worthwhile.

## Вывод и ограничения

Полный канонический датасет отзывов соответствует зарегистрированному физическому контракту `amazon_review_v1`, а реальные логические записи проходят строгую Pydantic-модель. Метаданные Amazon-товаров преобразуются в `amazon_product_v1`, а контракт обогащенного отзыва защищает соединение по `parent_asin`. Новый выборочный файл не создавался.

Полный версионный каталог товаров и реестр категорий уже построены и проверены в ноутбуках 04 и 05. Отдельный полный файл обогащенных отзывов пока не сохраняется, потому что соединение выполняется детерминированно; позднее требования офлайн-обработки ML покажут, оправдано ли создание еще одной полной копии данных.